In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import glob
import os

# 가장 최근 로그 자동 선택 (직접 지정하려면 아래 주석 해제)
log_dir = os.path.dirname(os.path.abspath('.'))
files = sorted(glob.glob('dataLog_data_*.csv'))
csv_filename = files[-1] if files else 'dataLog_data.csv'
# csv_filename = 'dataLog_data_20251206_195505.csv'  # 직접 지정

print(f'Loading: {csv_filename}')
df = pd.read_csv(csv_filename, sep=';')
print(df.columns.tolist())
print(f'Duration: {df["time"].iloc[-1]:.2f}s  |  Samples: {len(df)}')
df.head()

In [ ]:
# ── MPC 상태 vs 목표 ──────────────────────────────────────────
fig, axes = plt.subplots(4, 1, figsize=(12, 12), sharex=True)
t = df['time']

axes[0].plot(t, df['x_int'],    label='x_int (state)',  color='steelblue')
axes[0].plot(t, df['xRef_x'],   label='xRef_x (target)', color='steelblue', linestyle='--')
axes[0].set_ylabel('Position (m)'); axes[0].set_title('Position x'); axes[0].legend(); axes[0].grid(True)

axes[1].plot(t, np.degrees(df['pitch']),      label='pitch (state)',  color='tomato')
axes[1].plot(t, np.degrees(df['xRef_theta']), label='xRef_θ (target)', color='tomato', linestyle='--')
axes[1].set_ylabel('Angle (deg)'); axes[1].set_title('Pitch θ'); axes[1].legend(); axes[1].grid(True)

axes[2].plot(t, df['vel'],      label='vel (state)',   color='seagreen')
axes[2].plot(t, df['xRef_dx'], label='xRef_dx (target)', color='seagreen', linestyle='--')
axes[2].set_ylabel('Vel (m/s)'); axes[2].set_title('Forward velocity'); axes[2].legend(); axes[2].grid(True)

axes[3].plot(t, df['ang_vel_y'], label='ang_vel_y', color='purple')
axes[3].set_ylabel('rad/s'); axes[3].set_title('Pitch rate'); axes[3].legend(); axes[3].grid(True)
axes[3].set_xlabel('Time (s)')

plt.tight_layout(); plt.show()

In [ ]:
# ── 제어 출력 (토크) ──────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
t = df['time']

axes[0].plot(t, df['u_cmd'], label='u_cmd (MPC out)', color='navy')
axes[0].set_ylabel('Torque (Nm)'); axes[0].set_title('u_cmd (공통 토크)'); axes[0].legend(); axes[0].grid(True)

axes[1].plot(t, df['u_L'], label='u_L', color='darkorange')
axes[1].plot(t, df['u_R'], label='u_R', color='dodgerblue')
axes[1].set_ylabel('Torque (Nm)'); axes[1].set_title('좌/우 최종 토크'); axes[1].legend(); axes[1].grid(True)

axes[2].plot(t, df['u_L'] - df['u_R'], label='u_L - u_R (yaw diff)', color='crimson')
axes[2].set_ylabel('Torque (Nm)'); axes[2].set_title('Yaw 토크 차이'); axes[2].legend(); axes[2].grid(True)
axes[2].set_xlabel('Time (s)')

plt.tight_layout(); plt.show()

In [ ]:
# ── IMU 자세각 ────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
t = df['time']

axes[0].plot(t, np.degrees(df['roll']),      color='r', label='roll')
axes[0].set_ylabel('deg'); axes[0].set_title('Roll'); axes[0].legend(); axes[0].grid(True)

axes[1].plot(t, np.degrees(df['pitch_imu']), color='g', label='pitch')
axes[1].set_ylabel('deg'); axes[1].set_title('Pitch'); axes[1].legend(); axes[1].grid(True)

axes[2].plot(t, np.degrees(df['yaw']),       color='b', label='yaw')
axes[2].set_ylabel('deg'); axes[2].set_title('Yaw'); axes[2].legend(); axes[2].grid(True)
axes[2].set_xlabel('Time (s)')

plt.tight_layout(); plt.show()

In [ ]:
# ── 속도 지령 ────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
t = df['time']

axes[0].plot(t, df['v_ref_cmd'], label='v_ref_cmd', color='navy')
axes[0].set_ylabel('m/s'); axes[0].set_title('속도 지령'); axes[0].legend(); axes[0].grid(True)

axes[1].plot(t, df['v_L_ref'], label='v_L_ref', color='darkorange')
axes[1].plot(t, df['v_R_ref'], label='v_R_ref', color='dodgerblue')
axes[1].set_ylabel('m/s'); axes[1].set_title('좌/우 바퀴 속도 지령'); axes[1].legend(); axes[1].grid(True)
axes[1].set_xlabel('Time (s)')

plt.tight_layout(); plt.show()

In [ ]:
# ── CoM 위치 ─────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
t = df['time']

axes[0].plot(t, df['com_x'], color='r', label='com_x')
axes[0].set_ylabel('m'); axes[0].set_title('CoM X'); axes[0].legend(); axes[0].grid(True)

axes[1].plot(t, df['com_y'], color='g', label='com_y')
axes[1].set_ylabel('m'); axes[1].set_title('CoM Y'); axes[1].legend(); axes[1].grid(True)

axes[2].plot(t, df['com_z'], color='b', label='com_z')
axes[2].set_ylabel('m'); axes[2].set_title('CoM Z (높이)'); axes[2].legend(); axes[2].grid(True)
axes[2].set_xlabel('Time (s)')

plt.tight_layout(); plt.show()

In [ ]:
# ── 모드별 구간 표시 (전체 오버뷰) ──────────────────────────
mode_colors = {'INIT': 'gray', 'GO': 'green', 'BACK': 'red', 'LEFT': 'blue', 'RIGHT': 'orange'}
t = df['time'].values

fig, ax = plt.subplots(figsize=(14, 3))
prev_mode = df['mode'].iloc[0]
seg_start = t[0]

for i in range(1, len(df)):
    cur_mode = df['mode'].iloc[i]
    if cur_mode != prev_mode or i == len(df) - 1:
        ax.axvspan(seg_start, t[i], alpha=0.3,
                   color=mode_colors.get(prev_mode, 'white'),
                   label=prev_mode)
        seg_start = t[i]
        prev_mode = cur_mode

ax.plot(t, df['pitch'], color='black', linewidth=0.8, label='pitch')
handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax.legend(by_label.values(), by_label.keys())
ax.set_xlabel('Time (s)'); ax.set_ylabel('pitch (rad)')
ax.set_title('Mode 구간 + Pitch 오버뷰')
ax.grid(True)
plt.tight_layout(); plt.show()